# Train and Test: The Split That Makes a Number Mean Something

**Computational Text Analysis Pack · Module 07 — Classification**

*Technique:* Held-out evaluation; leakage; the majority-class baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/06-supervised-classification/07a_train_test_on_text.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Classification asks: given the words, can a model recover a label you already
have? It sounds circular, and for prediction it often is. For DH its value is
different — **if a model can tell two kinds of writing apart, the difference is
real and measurable, and the model will tell you which words carry it.**

Before any of that, one discipline. A model tested on the text it learned from
scores brilliantly and means nothing. Everything here rests on the split.

## The tooling

**scikit-learn** end to end: `train_test_split`, `LogisticRegression`,
`LinearSVC`, `MultinomialNB`, `cross_val_score`. This is the library's core
competence and the reason it is the general practitioner of the Python text
stack.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

LANGUAGE = "English"        # ← change this
MIN_DOCS = 40               # drop categories too small to learn or evaluate

sub = df[df["language"] == LANGUAGE]
keep = sub["category"].value_counts()
sub = sub[sub["category"].isin(keep[keep >= MIN_DOCS].index)].reset_index(drop=True)
print(sub["category"].value_counts().to_string())

## The baseline you must beat

Always compute it first. If 60% of documents are politics, a model that guesses
"politics" every time scores 60% — and an announced accuracy of 65% is barely
better than a constant.

In [ ]:
baseline = sub["category"].value_counts(normalize=True).iloc[0]
print(f"majority class: {sub['category'].value_counts().index[0]}")
print(f"baseline accuracy (always guess it): {baseline:.1%}")
print()
print("Any accuracy you report is only meaningful next to this number.")

In [ ]:
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    sub["text"], sub["category"], test_size=0.25, random_state=0,
    stratify=sub["category"])

print(f"train: {len(X_train_txt):,} documents")
print(f"test : {len(X_test_txt):,} documents")
print()
print("stratify= keeps the class proportions the same in both halves. Without")
print("it a small category can land entirely in one side and vanish.")

## The mistake that inflates every published accuracy

Fitting the vectoriser on **all** the text before splitting lets the model see
test-set vocabulary during training. This is **leakage**, and it is easy to do by
accident because it looks tidier.

Fit on train, transform test. Never the other way.

In [ ]:
vec = TfidfVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=3, max_features=5000,
                      sublinear_tf=True)
X_train = vec.fit_transform(X_train_txt)     # fit ONLY on training text
X_test = vec.transform(X_test_txt)           # transform, do not fit

clf = LogisticRegression(max_iter=2000, C=1.0)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

acc = accuracy_score(y_test, pred)
print(f"accuracy on held-out documents: {acc:.3f}")
print(f"baseline                      : {baseline:.3f}")
print(f"improvement over baseline     : {acc - baseline:+.3f}")

In [ ]:
# The leaky version, so you can see the size of the exaggeration.
vec_leaky = TfidfVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=3, max_features=5000,
                            sublinear_tf=True)
X_all = vec_leaky.fit_transform(sub["text"])          # fitted on everything
Xtr, Xte, ytr, yte = train_test_split(X_all, sub["category"], test_size=0.25,
                                      random_state=0, stratify=sub["category"])
leaky = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
print(f"leaky accuracy   : {accuracy_score(yte, leaky.predict(Xte)):.3f}")
print(f"correct accuracy : {acc:.3f}")
print()
print("The gap here is small because vocabulary leakage is mild. It is not")
print("always small — and with any preprocessing step fitted on all the data")
print("(scaling, feature selection, topic models) it can be enormous.")

## Accuracy is not enough

One number hides which classes the model gets right. The confusion matrix does
not.

In [ ]:
labels = sorted(sub["category"].unique())
cm = confusion_matrix(y_test, pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true {l}" for l in labels],
                     columns=[f"pred {l}" for l in labels])
print(cm_df.to_string())
print()
print(classification_report(y_test, pred, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
norm = cm / cm.sum(axis=1, keepdims=True)
im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{norm[i, j]:.2f}", ha="center", va="center",
                color="white" if norm[i, j] > 0.5 else "black", fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Row-normalised confusion matrix")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

### 🔧 Your turn 1

Find the two categories the model confuses most.

Read three documents it got wrong. Are they genuinely ambiguous — a story that is
both politics and business — or is the label itself doubtful? Misclassifications
are the most informative output of a classifier for humanities work.

### 🔧 Your turn 2

Change `LANGUAGE` to Hausa or Yoruba and re-run.

Accuracy will usually be lower. Before concluding the languages are "harder":
check the number of training documents and the number of categories. Those two
explain most cross-language differences in this corpus.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Politics/business and politics/health are the usual confusions,
and reading the errors normally shows they are real overlaps rather than model
failures — a story about a health budget is both. This matters for how you write
it up: an 80%-accurate classifier whose errors are all genuinely ambiguous
documents is telling you the categories overlap, which is a finding about the
taxonomy, not about the model.

**Your turn 2.** Fewer training documents means lower accuracy regardless of
language, and this corpus caps most languages at 400–900 documents. The way to
make a fair comparison is to sample every language down to the same number of
documents and the same set of categories — then any remaining gap is about the
language or the writing rather than the sample. Doing that comparison properly
would be a good short paper.

</details>

## Where this stops

You can evaluate a model honestly. Next: **07b**, on what makes accuracy go up or
down, and how much of it is your choices rather than the text.